In [1]:

import os
import sys
import subprocess
import shutil
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------
REPO = Path("/kaggle/working/filament-seg")
OUTPUT = Path("/kaggle/working/filament_outputs")
YOLO_DATA = Path("/kaggle/working/filament_yolo_data")

DATA_ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

OUTPUT.mkdir(parents=True, exist_ok=True)
YOLO_DATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# VERIFY DATA
# --------------------------------------------------
assert (DATA_ROOT / "train/train_images").is_dir(), "Training images missing"
assert (DATA_ROOT / "test/test_images").is_dir(), "Test images missing"
assert (
    DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
).is_file(), "Annotation JSON missing"

print("Dataset verified:", DATA_ROOT)

# --------------------------------------------------
# INSTALL DEPENDENCIES
# DO NOT reinstall torch or torchvision
# --------------------------------------------------
packages = [
    "segmentation-models-pytorch==0.5.0",
    "timm==1.0.29",
    "albumentations==2.0.8",
    "pycocotools==2.0.10",
    "scipy==1.15.3",
    "pandas==2.3.3",
    "tqdm==4.67.1",
    "matplotlib==3.10.6",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# --------------------------------------------------
# VERIFY PYTORCH AND REQUIRED IMPORTS
# --------------------------------------------------
import torch
import segmentation_models_pytorch as smp
import timm
import albumentations
import pycocotools

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SMP:", smp.__version__ if hasattr(smp, "__version__") else "import OK")

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings"

# --------------------------------------------------
# CLONE OR REPAIR REPOSITORY
# --------------------------------------------------
required_files = [
    "scripts/preprocess.py",
    "scripts/make_splits.py",
    "scripts/train.py",
    "scripts/export_yolo.py",
    "scripts/train_detector.py",
]

valid_repo = (
    (REPO / ".git").exists()
    and all((REPO / f).is_file() for f in required_files)
)

if not valid_repo:
    if REPO.exists():
        backup = Path("/kaggle/working/filament-seg-old")
        if backup.exists():
            shutil.rmtree(backup)
        shutil.move(str(REPO), str(backup))

    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ameyypawar/filament-seg.git",
            str(REPO)
        ],
        check=True
    )

assert all((REPO / f).is_file() for f in required_files)

# --------------------------------------------------
# ENVIRONMENT
# --------------------------------------------------
os.environ["FILAMENT_DATA_ROOT"] = str(DATA_ROOT)
os.environ["FILAMENT_OUTPUT_ROOT"] = str(OUTPUT)
os.environ["PYTHONUNBUFFERED"] = "1"

workers = min(4, os.cpu_count() or 2)

def run_command(command, cwd=REPO):
    print("\nRUNNING:", " ".join(map(str, command)))
    subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        check=True
    )

os.chdir(REPO)

print("\nCELL 1 SUCCESS")
print("Repository:", REPO)
print("Output:", OUTPUT)
print("Workers:", workers)
print("GPU:", torch.cuda.get_device_name(0))


Dataset verified: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


PyTorch: 2.11.0+cu130
CUDA available: True
SMP: 0.5.0


Cloning into '/kaggle/working/filament-seg'...



CELL 1 SUCCESS
Repository: /kaggle/working/filament-seg
Output: /kaggle/working/filament_outputs
Workers: 4
GPU: Tesla T4


In [2]:

import os
import sys
from pathlib import Path

os.chdir(REPO)

# Re-establish environment variables
os.environ["FILAMENT_DATA_ROOT"] = str(DATA_ROOT)
os.environ["FILAMENT_OUTPUT_ROOT"] = str(OUTPUT)

print("Starting preprocessing...")

run_command([
    sys.executable,
    "scripts/preprocess.py",
    "--workers", str(workers)
])

print("\nCreating date-based validation split...")

run_command([
    sys.executable,
    "scripts/make_splits.py",
    "--group-by", "date"
])

CACHE = REPO / "data" / "cache"
SPLIT = OUTPUT / "splits.json"

assert CACHE.exists(), f"Cache not found: {CACHE}"
assert SPLIT.exists(), f"Split not found: {SPLIT}"

print("\nCELL 2 SUCCESS")
print("Cache:", CACHE)
print("Split:", SPLIT)


Starting preprocessing...

RUNNING: /usr/bin/python3 scripts/preprocess.py --workers 4
887 observations (707 train, 180 test)
  25/887
  50/887
  75/887
  100/887
  125/887
  150/887
  175/887
  200/887
  225/887
  250/887
  275/887
  300/887
  325/887
  350/887
  375/887
  400/887
  425/887
  450/887
  475/887
  500/887
  525/887
  550/887
  575/887
  600/887
  625/887
  650/887
  675/887
  700/887
  725/887
  750/887
  775/887
  800/887
  825/887
  850/887
  875/887
  887/887
wrote 887 disk geometries -> /kaggle/working/filament-seg/data/cache/disk.json
wrote 1154 new mask PNGs -> /kaggle/working/filament-seg/data/cache/mask
wrote 707 new consensus PNGs -> /kaggle/working/filament-seg/data/cache/consensus

Creating date-based validation split...

RUNNING: /usr/bin/python3 scripts/make_splits.py --group-by date
group_by=date  groups=656
train: 906 views / 563 observations
val  : 248 views / 144 observations
observation overlap between train and val: 0 (must be 0)
wrote /kaggle/working

In [3]:

import os
import sys
import time
import subprocess
from pathlib import Path

os.chdir(REPO)

unet_checkpoint = OUTPUT / "unet_resnet34.pt"
unet_log = OUTPUT / "unet_training.log"

assert torch.cuda.is_available()
assert (REPO / "data/cache").exists()
assert (OUTPUT / "splits.json").exists()

command = [
    sys.executable, "scripts/train.py",
    "--epochs", "30",
    "--batch-size", "16",
    "--crop-size", "512",
    "--encoder", "resnet34",
    "--lr", "0.001",
    "--crops-per-image", "8",
    "--workers", str(workers),
    "--limit-val", "48",
    "--preload",
    "--seed", "0",
    "--target", "annotator",
    "--views", "one",
    "--out", str(unet_checkpoint)
]

print("Starting U-Net training on:", torch.cuda.get_device_name(0))
print("Maximum allowed time: 4 hours 45 minutes")
print("Log:", unet_log)

start = time.time()

try:
    with open(unet_log, "w") as log:
        result = subprocess.run(
            command,
            cwd=str(REPO),
            env=os.environ.copy(),
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=17100,
            check=False
        )

    if result.returncode != 0:
        print("\nTraining failed. Last log lines:")
        print("\n".join(unet_log.read_text(errors="replace").splitlines()[-60:]))
        raise RuntimeError(f"U-Net training exited with code {result.returncode}")

except subprocess.TimeoutExpired:
    print("U-Net reached its 4h45 time limit.")
    raise

print("Training time:", round((time.time()-start)/60, 1), "minutes")

print("\nCheckpoint files:")
for p in OUTPUT.rglob("*.pt"):
    print(p, round(p.stat().st_size / 1024**2, 2), "MB")

assert (
    unet_checkpoint.exists()
    or (OUTPUT / "unet_resnet34_last.pt").exists()
), "Training ended without producing a checkpoint"

print("\nCELL 3 SUCCESS")


Starting U-Net training on: Tesla T4
Maximum allowed time: 4 hours 45 minutes
Log: /kaggle/working/filament_outputs/unet_training.log
Training time: 103.7 minutes

Checkpoint files:
/kaggle/working/filament_outputs/unet_resnet34_last.pt 93.37 MB
/kaggle/working/filament_outputs/unet_resnet34.pt 93.37 MB

CELL 3 SUCCESS


In [4]:

import sys
import os

os.chdir(REPO)

assert (OUTPUT / "splits.json").exists()
assert (REPO / "data/cache").exists()

run_command([
    sys.executable,
    "scripts/export_yolo.py",
    "--out", str(YOLO_DATA)
])

yaml_file = YOLO_DATA / "data.yaml"

assert yaml_file.exists(), f"YOLO data.yaml missing: {yaml_file}"

print("CELL 4 SUCCESS")
print("YOLO dataset:", YOLO_DATA)



RUNNING: /usr/bin/python3 scripts/export_yolo.py --out /kaggle/working/filament_yolo_data
train: 906 views, 6342 filaments
val: 248 views, 1857 filaments
wrote /kaggle/working/filament_yolo_data/data.yaml
CELL 4 SUCCESS
YOLO dataset: /kaggle/working/filament_yolo_data


In [5]:

import sys
import subprocess
import os
from pathlib import Path

# Install detector dependency without replacing PyTorch
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "ultralytics==8.4.173"
    ],
    check=True
)

import ultralytics

print("Ultralytics:", ultralytics.__version__)

yaml_file = YOLO_DATA / "data.yaml"
detector_output = OUTPUT / "detector"

assert yaml_file.exists(), "YOLO dataset is missing"
detector_output.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, "scripts/train_detector.py",
    "--data", str(yaml_file),
    "--out", str(detector_output),
    "--model", "yolo11s-seg.pt",
    "--imgsz", "1024",
    "--epochs", "60",
    "--batch", "8",
    "--device", "0,1",
    "--workers", str(workers),
    "--patience", "20",
    "--seed", "0"
]

print("Starting YOLO11s training...")

log_path = OUTPUT / "detector_training.log"

with open(log_path, "w") as log:
    result = subprocess.run(
        command,
        cwd=str(REPO),
        env=os.environ.copy(),
        stdout=log,
        stderr=subprocess.STDOUT,
        timeout=13500,
        check=False
    )

if result.returncode != 0:
    print("\nDetector training error:")
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-60:]))
    raise RuntimeError(f"Detector training exited with code {result.returncode}")

print("CELL 5 SUCCESS")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.173
Starting YOLO11s training...
CELL 5 SUCCESS


In [6]:

from pathlib import Path
import json

print("=== ALL CHECKPOINTS ===")

checkpoints = list(OUTPUT.rglob("*.pt"))

for p in checkpoints:
    print(p, round(p.stat().st_size / 1024**2, 2), "MB")

assert checkpoints, "No model checkpoints found!"

manifest = {
    "repository": str(REPO),
    "data_root": str(DATA_ROOT),
    "output": str(OUTPUT),
    "checkpoints": [
        {
            "path": str(p),
            "size_mb": round(p.stat().st_size / 1024**2, 2)
        }
        for p in checkpoints
    ]
}

manifest_path = OUTPUT / "training_manifest.json"

manifest_path.write_text(json.dumps(manifest, indent=2))

print("\nManifest saved:", manifest_path)
print("Training artifacts verified.")


=== ALL CHECKPOINTS ===
/kaggle/working/filament_outputs/unet_resnet34_last.pt 93.37 MB
/kaggle/working/filament_outputs/unet_resnet34.pt 93.37 MB
/kaggle/working/filament_outputs/detector/detector_last.pt 19.64 MB
/kaggle/working/filament_outputs/detector/detector_best.pt 19.64 MB
/kaggle/working/filament_outputs/detector/train/weights/best.pt 19.64 MB
/kaggle/working/filament_outputs/detector/train/weights/last.pt 19.64 MB

Manifest saved: /kaggle/working/filament_outputs/training_manifest.json
Training artifacts verified.
